## Visual Discovery Analysis 
### Introduction 
- Launched: 2025-12-01 
- Layer: 06
- Groups: 
    - Test: 2, 3, 16, 24, 25, 27, 37, 45, 59, 65, 71, 74, 87, 98, 106, 126
    - Control: 6, 10, 19, 35, 38, 58, 60, 61, 77, 86, 102, 104, 110, 112, 122, 123
  formatted: 
      case when extended_signup_segments_v3 in ('002', '003', '016', '024', '025', '027', '037', '045', '059', '065', '071', '074', '087', '098', '106', '126') then 'Test'
           when extended_signup_segments_v3 IN ('006', '010', '019', '035', '038', '058', '060', '061', '077', '086', '102', '104', '110', '112', '122', '123') then 'Control'
           ELSE 'Other' 
      end
- design within IMU 
  Women:
       Total: 10 cards
             Looks: 5 cards
             Trends: 4
             Thematic_collection: 1
  Men:
        Total: 5 cards
             Looks: 0 cards
              Trends: 3
              Aesthetics: 1
              Thematic_collection:  1


## Iteration 2 

- period: 1/7 - 1/23
- Layer: 09
- Test: 42, 43, 59, 61, 63, 65, 69, 70, 78, 79, 89, 100, 107, 109, 117, 126
- Control: 26, 44, 46, 56, 66, 67, 72, 76, 84, 92, 95, 101, 105, 112, 118, 123
#### Design 
Control
     P0: Retargeting (Cross merch/Recent brand/Top 8)
     P1: Retargeting 
     P2: Shows 50% and trending search 50% 
     P3: Retargeting 
     P4: Retargeting 
     P5: Brand Expansion 
Test
     ​​P0: Retargeting (Cross merch/Recent brand/Top 8)
     P1: Retargeting 
     P2: Shows 50% and trending search 50% 
     P3: Retargeting 
     P4: IMU 
     P5: Brand Expansion
## Iteraction 3 - Personalization Matching look 

ABC test

Layer : L06
IMU Programmed: 1, 6, 12, 19, 28, 40, 64, 66, 80, 82, 92, 99, 100, 102, 107, 124

IMU Personalized: 2, 14, 15, 23, 24, 35, 46, 57, 63, 67, 70, 104, 112, 119, 125, 127

Control: 3, 9, 16, 17, 18, 21, 30, 32, 51, 54, 73, 74, 75, 76, 78, 105


    
    

### 1.0 Feed Activity 

- rank is used to pull all the collection_id related informatino from feed IMU; 

In [0]:
#install  /dbfs/FileStore/shared_uploads/dmat/dmat-0.1-py3-none-any.whl

In [0]:
%pip install /dbfs/FileStore/shared_uploads/dmat/pmgoogle-1.4-py3-none-any.whl
%pip install  /dbfs/FileStore/shared_uploads/dmat/dmat-0.1-py3-none-any.whl

dbutils.widgets.text(
    "report_run_date",      # Widget name
    "",                     # Default value
    "Report Run Date"       # Optional label
)

In [0]:
from dmat.helpers import *
from dmat import spark_helpers as dsh

from dateutil.relativedelta import relativedelta
import datetime

spark.conf.set("spark.sql.shuffle.partitions","1024")
spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")
#sc.hadoopConfiguration.set("mapreduce.fileoutputcommitter.marksuccessfuljobs", "false")

##############
## Imports & Inits
import pyspark
from pyspark import *
from pyspark.sql import *
from dateutil import tz
from dateutil.relativedelta import relativedelta
import datetime

from pmgoogle.sheets import GoogleSheet
import numpy as np
import pandas as pd
import time
import json
import pyspark.sql.functions as F

# get service account creds
#athena-master@athena-284716.iam.gserviceaccount.com
#service_account = json.loads(dbutils.secrets.get(scope="data_pa-dmat_dmat-secrets-scope", key="athena_sa"))['general']  

sc = spark._sc
helpers = sc._jvm.com.poshmark.spark.helpers
Config = helpers.Config
Redshift = helpers.Redshift
SaveMode = sc._jvm.org.apache.spark.sql.SaveMode
SaveMode_Append = SaveMode.Append
SaveMode_Overwrite = SaveMode.Overwrite
SaveMode_Ignore = SaveMode.Ignore
SaveMode_Error = SaveMode.ErrorIfExists

## Load default configs for helper libraries and register UDFs
Config.reloadConfigs(sc._jsc.sc())
Config.registerUDFs()


# Returns the current UTC time
def getUTC():
    return datetime.datetime.utcnow().replace(tzinfo=tz.gettz('UTC'))

# Returns the current time in 'US/Pacific'
def getPacificTime(date=False, month=False):
    now = getUTC().astimezone(tz.gettz('US/Pacific'))
    if date:
        now = now.date()
    if month:
        now = now.replace(day=1)

    return now



dsh.init(spark)
Redshift = sc._jvm.com.poshmark.spark.helpers.Redshift

def get_widget(widget_name, default="", _type=str):
    try:
        tmp =  _type(dbutils.widgets.get(widget_name))
        if tmp:
            return tmp
        else:
            return default
    except:
        return default

run_now = get_widget('run_now',default='false')
if (run_now.lower() == 'true'):
    force = True
else:
    force = False

send_day = getPacificTime().isoweekday()
start_of_month = True if getPacificTime().day == 1 else False

## Get today & yesterday
local_date = getPacificTime() 
today = local_date.strftime('%Y-%m-%d')
yesterday = (local_date - datetime.timedelta(days=1)).strftime('%Y-%m-%d')

##############
## get a value from a widget
report_run_date = dbutils.widgets.get("report_run_date")
process_end_date = get_widget('report_run_date', default=yesterday)
## last_90_days = (datetime.datetime.strptime(process_end_date, "%Y-%m-%d") - datetime.timedelta(days=90)).strftime('%Y-%m-%d') 
last_90_days = (datetime.datetime.strptime(process_end_date, "%Y-%m-%d") - datetime.timedelta(days=3)).strftime('%Y-%m-%d')
## last_90_days = (datetime.datetime.strptime(process_end_date, "%Y-%m-%d") - datetime.timedelta(days=2)).strftime('%Y-%m-%d')
process_start_date = last_90_days

## process_date = get_widget("process_date",default=yesterday)
## process_end_date = get_widget("process_end_date",default=yesterday)

# print(f"process_date: {process_date}, process_end_date: {process_end_date}")

# stage_s3_path = "s3://data-tmp-poshmark-production/dmat/project_highway/top_stats/v2/"

def to_jvm_list(py_list):
    l = len(py_list)
    jvm_list = sc._gateway.new_array(sc._jvm.java.lang.String, l)
    for i in range(l):
        jvm_list[i] = py_list[i]
    return jvm_list
print(process_start_date, process_end_date)


In [0]:
queryDF = spark.sql(f""" 

select extended_signup_segments_v3, event_date, user_id, home_domain, user_gender, event_at, app_type, verb, dob_name, on_name, story_type, unit_position, new_content_position, content_position, rank, collection_id, sub_unit_type
from (
SELECT *,
    posexplode(from_json(tracking_meta, 'STRUCT<merch_units:ARRAY<STRUCT<collection_id:STRING, sub_unit_type:STRING>>>').merch_units) AS (rank, merch_unit),
    merch_unit.collection_id,
    merch_unit.sub_unit_type

from (
select 
r.event_date,
r.actor.id as user_id,
r.using.app_type as app_type,
dw_users.home_domain, 
CASE WHEN dw_users.gender in ('male','male (hidden)') THEN 'male'
            ELSE 'female' END  AS user_gender,
REPLACE(CAST(dw_users.extended_signup_segments_v3.L06 AS STRING), '"', '') as extended_signup_segments_v3,
f_pm_epoch_to_pacific(r.at) as event_at,
r.verb,
r.direct_object.name as dob_name,
r.on.name as on_name,
r.properties.story_type as story_type,
r.properties.unit_position as unit_position,
case when r.using.app_type = 'web' then r.properties.content_position - 1 else r.properties.content_position end as new_content_position,
r.properties.content_position as content_position,
r.properties.tracking_meta as tracking_meta,
r.properties.content_type as content_type

from raw_events r
inner join s3_analytics.dw_users dw_users on r.actor.id = dw_users.user_id

WHERE 
((r.verb = 'click' AND r.on.name = 'feed' AND r.on.tab = 'self' AND r.properties.story_type in ('pv2_inspirational_merch_MIFU') AND r.properties.unit_position >= 0)
OR 
(r.verb = 'observe' and r.on.name = 'feed' AND r.on.tab = 'self' AND r.properties.story_type in ('pv2_inspirational_merch_MIFU') AND r.properties.unit_position >= 0)
)

    AND r.event_date between '{process_start_date}' and '{process_end_date}' 
    AND r.actor.type IN ('user') 
    AND ((r.using.app_type IN ('iphone', 'ipad', 'android','web')))
    AND dw_users.home_domain in ('us','ca') and dw_users.is_valid_user is true
    AND (NOT coalesce((datediff((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END),
            coalesce(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))
    --AND r.actor.id in ('int1:7f7a105f-3077-4a83-86ac-95fccc164fd4')
) K
) P
where rank = new_content_position
""")
#val s3_path = "s3://data-tmp-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl/"
#val s3_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl/"
#queryDF.write.format("parquet").mode("Overwrite").save(s3_path)

In [0]:
output_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl/"

queryDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("visual_discovery_feed_content_lvl")

##print(process_start_date)
##print(process_end_date)
print(output_path)



s3_path_old = "s3://data-tmp-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl/"
s3_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl/"
df = spark.read.format("parquet").load(s3_path_old)
df.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)


In [0]:
#s3_path = "s3://data-tmp-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl/"
s3_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl/"
df = spark.read.format("parquet").load(s3_path)
df.createOrReplaceTempView("visual_discovery_feed_content_lvl")

In [0]:
# %sql 
# select min(event_Date), max(event_Date) from visual_discovery_feed_content_lvl


### 2.0 Enrichment of the collections Information and Department 

In [0]:
queryDF = spark.sql(f""" 
WITH col AS (
   
    SELECT 
        t1._id AS collection_id,
        t1.nm AS collection_name,
        t1.asthc[0] as aesthetic_id, 
        t1.depts[0] AS department_id,
        t3.dsp AS department_name
    FROM s3_raw_mongo.collections t1
    LEFT JOIN (
        SELECT DISTINCT 
            _id,
            dsp
        FROM s3_raw_mongo.catalog_entries
        WHERE typ = 'd'
    ) t3 
        ON t3._id = t1.depts[0]
),

ast AS (
 
    SELECT
        t2._id AS aesthetic_id,
        t2.nm AS aesthetic_name,
        t2.dpt[0] AS department_id,
        t3.dsp AS department_name
    FROM s3_raw_mongo.aesthetics t2
    LEFT JOIN (
        SELECT DISTINCT 
            _id,
            dsp
        FROM s3_raw_mongo.catalog_entries
        WHERE typ = 'd'
    ) t3
        ON t3._id = t2.dpt[0]
)

SELECT

    k.*,

    CASE 
        WHEN k.sub_unit_type = 'aesthetic' THEN ast.aesthetic_name
        WHEN k.sub_unit_type  =  'trend'      THEN col.collection_name
        WHEN k.sub_unit_type   = 'thematic_collection'   THEN col.collection_name
        ELSE NULL
    END AS unit_name,

    col.collection_name as collection_name,
    case when k.sub_unit_type   =  'thematic_collection'   then thm.aesthetic_name 
        else ast.aesthetic_name end as aesthetic_name,

    CASE 
        WHEN k.sub_unit_type = 'aesthetic' THEN ast.department_name
        WHEN k.sub_unit_type = 'thematic_collection' THEN thm.department_name
        WHEN k.sub_unit_type  =  'trend'  THEN col.department_name
    END AS department_name

FROM visual_discovery_feed_content_lvl k -- this has all the data for now

LEFT JOIN col 
    ON k.collection_id = col.collection_id

LEFT JOIN ast  -- if aesthetic then using asethetic id  join asethetic  ID 
    ON k.collection_id = ast.aesthetic_id
Left join ast thm 
    on col.aesthetic_id = thm.aesthetic_id
where  1=1 AND event_date between '{process_start_date}' and '{process_end_date}' 

 """)


In [0]:
output_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl2/"

queryDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("visual_discovery_feed_content_lvl2")

##print(process_start_date)
##print(process_end_date)
print(output_path)



In [0]:
output_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl2/"

#queryDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem.write.partitionBy("booked_date").format("parquet").mode("Overwrite").save(output_path)
## base_orderitem_df.write.format("parquet").mode("Overwrite").save(output_path)

## ## revert back to dynamic from static
## spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

look_sdf = spark.read.parquet(output_path)
look_sdf.createOrReplaceTempView("visual_discovery_feed_content_lvl2")

##print(process_start_date)
##print(process_end_date)
print(output_path)



In [0]:
# %sql 
# select min(event_Date), max(event_date) 
# from visual_discovery_feed_content_lvl2
# limit 5

In [0]:
### Move to 1 Year production Table 


s3_path_old = "s3://data-tmp-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl/" # 1 month Bucket
s3_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl/" #one year bucket
df = spark.read.format("parquet").load(s3_path_old)
df.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)


s3_path_old = "s3://data-tmp-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl2/"
s3_path = "s3://analytics-scratch-reviewed-poshmark-production/qiong/for_you_feed/visual_discovery/visual_discovery_feed_content_lvl2/"
df = spark.read.format("parquet").load(s3_path_old)
df.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)


In [0]:
# %sql 
# select event_date, 
# count(case when verb = 'observe' then user_id else null end) as client_impressions,
# count(case when verb = 'click' then user_id else null end) as clicks,
# count(distinct case when verb = 'observe' then user_id else null end) as client_impressions_users,
# count(distinct case when verb = 'click' then user_id else null end) as clicks_users
#  from visual_discovery_feed_content_lvl2
# where event_date >= '2026-01-08' 
# group by 1
# order by 1

### 2.1 Aggregated the data for Analysis 


case when extended_signup_segments_v3 in 
in ('001', '006', '012', '019', '028', '040', '064', '066', '080', '082', '092', '099', '100', '102', '107', '124'
) then 'Test_NonPersonalized'
when extended_signup_segments_v3 in 
in ('002', '014', '015', '023', '024', '035', '046', '057', '063', '067', '070', '104', '112', '119', '125', '127'
) then 'Test_Personalized'
when extended_signup_segments_v3 in 
in ('003', '009', '016', '017', '018', '021', '030', '032', '051', '054', '073', '074', '075', '076', '078', '105') then 'Control' 
else 'Other'
end as exp_grp, 
  

Old - 
Test ['042', '043', '059', '061', '063', '065', '069', '070', '078', '079', '089', '100', '107', '109', '117', '126']
Control ['026', '044', '046', '056', '066', '067', '072', '076', '084', '092', '095', '101', '105', '112', '118', '123']

Test: 42, 43, 59, 61, 63, 65, 69, 70, 78, 79, 89, 100, 107, 109, 117, 126
Control: 26, 44, 46, 56, 66, 67, 72, 76, 84, 92, 95, 101, 105, 112, 118, 123

Old  - 

CASE 
    WHEN extended_signup_segments_v3 IN 
     ('002', '003', '016', '024', '025', '027', '037', '045', '059', '065', '071', '074', '087', '098', '106', '126') then 'Test'
    WHEN extended_signup_segments_v3 IN ('006', '010', '019', '035', '038', '058', '060', '061', '077', '086', '102', '104', '110', '112', '122', '123') then 'Control'
    ELSE 'Other' 
  END AS exp_grp,




In [0]:
## updated on 2/3 for user segment analysis 

In [0]:
# Incremental 
## Redshift - Minium Dataset; 

In [0]:
%sql
create or replace temporary view visual_discovery_feed_content_lvl2_agg
as 
select 
event_date, 
unit_position,
story_type, 
user_id, 
home_domain, 
user_gender, 
unit_name,
collection_name,
aesthetic_name,
department_name, 

case when extended_signup_segments_v3 
in ('001', '006', '012', '019', '028', '040', '064', '066', '080', '082', '092', '099', '100', '102', '107', '124'
) then 'Test_NonPersonalized'
when extended_signup_segments_v3 
in ('002', '014', '015', '023', '024', '035', '046', '057', '063', '067', '070', '104', '112', '119', '125', '127'
) then 'Test_Personalized'
when extended_signup_segments_v3 
in ('003', '009', '016', '017', '018', '021', '030', '032', '051', '054', '073', '074', '075', '076', '078', '105') then 'Control' 
else 'Other'
end as exp_grp, 

content_position, 
new_content_position, 
sub_unit_type, 
app_type,

count(case when verb = 'observe' then user_id else null end) as client_impressions,
count(case when verb = 'click' then user_id else null end) as clicks,
count(distinct case when verb = 'observe' then user_id else null end) as client_impressions_users,
count(distinct case when verb = 'click' then user_id else null end) as clicks_users

from visual_discovery_feed_content_lvl2
where 1=1 --event_date between '2025-11-11' and '2025-11-17'
group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15

### Ignore -- 2.2 Validation using Feed CTR table 

 //Read from S3 directly via spark dataframe (table creation can be skipped)
var sdf = spark.read.parquet("s3://data-tmp-poshmark-production/karunya/for_you_feed/daily_feed_unit_position_level_ctr_raw_data/")
sdf.createOrReplaceTempView("daily_feed_unit_position_level_ctr_raw_data")

- within test group, we are seeing some of story type that are not pv2_inspirational_merch_MIFU which drives up the total clicks. 

In [0]:
#### Validation stoped 
### 3.0 Pull Data for analysis 

In [0]:
# %sql 
# select 
# event_date, 
# exp_grp, 
# unit_position,
# home_domain, 
# user_gender,

# sum(client_impressions) as client_impressions, 
# sum(clicks) as clicks, 
# sum(client_impressions_users) as client_impressions_users, 
# sum(clicks_users) as clicks_users
# from visual_discovery_feed_content_lvl2_agg
# where 1=1
# and event_Date between '2026-02-24' and '2026-03-08'
# and home_domain = 'us' and user_gender = 'female'
# group by 1,2,3,4,5

In [0]:
# %sql 

# select 
# event_date, 
# unit_position,
# story_type, 
# collection_name,
# aesthetic_name,
# department_name, 
# exp_grp, 
# content_position, 
# new_content_position, 
# sub_unit_type, 
# app_type,
# unit_name, 
# home_domain, 
# user_gender, 
# sum(client_impressions) as client_impressions, 
# sum(clicks) as clicks, 
# sum(client_impressions_users) as client_impressions_users, 
# sum(clicks_users) as clicks_users
# from visual_discovery_feed_content_lvl2_agg
# where 1=1
# and event_Date between '2026-02-23' and '2026-03-08'

# group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14


In [0]:
# %sql 
# select 
# event_date, 
# unit_position,
# exp_grp,  
# new_content_position, 
# sub_unit_type, 
# app_type, 
# sum(client_impressions) as client_impressions, 
# sum(clicks) as clicks, 
# sum(client_impressions_users) as client_impressions_users, 
# sum(clicks_users) as clicks_users
# from visual_discovery_feed_content_lvl2_agg
# where 1=1
# and event_date between '2026-01-09' and '2026-01-21'
# group by 1,2,3,4,5,6
